In [1]:
import re 
import ast
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path

from pre_processing_utils import Utils

In [36]:
HOURLY_PAYMENT = 17
SECONDS_PAYMENT = HOURLY_PAYMENT / (60 * 60)

In [37]:
folder_path = Path.cwd().parent / "data" / "new"

dict_worker_id = {}
for subdir in folder_path.iterdir():
    if subdir.is_dir():
        df = pd.read_csv(subdir / "Request.csv")
        worker_id_list = []
        for key, row in df.iterrows():
            value = row['unique_id']
            if isinstance(value, str):
                value_ = value.split(':')[0]
                if value_ not in worker_id_list:
                    worker_id_list.append(value_)
        dict_worker_id[subdir.name] = {
            idx  + 1: value for idx, value in enumerate(worker_id_list)
        }

#dict_worker_id

In [38]:
folder_path = Path.cwd().parent / "data/new" 

df = pd.DataFrame()
for subdir in folder_path.iterdir():
    if subdir.is_dir():
        if "new" not in subdir.name:
            print(f"Processing {subdir.name}...")
            df1 = pd.read_csv(subdir / "PersonalityTrial.csv")
            df1["session"] = subdir.name
            
            df = pd.concat([df, df1], ignore_index=True)

columns = ["session", "network_id", "participant_id", "failed", "failed_reason", "question", "answer", "time_taken"]
df = df[columns]
df.head()

Processing 05-25-batch-1...


,session,network_id,participant_id,failed,failed_reason,question,answer,time_taken
0,05-25-batch-1,8,1,False,NaN,Does a thorough job.,1.0,2.936
1,05-25-batch-1,5,1,False,NaN,Has few artistic interests.,2.0,1.294
2,05-25-batch-1,2,1,False,NaN,Is generally trusting.,3.0,1.165
3,05-25-batch-1,1,1,False,NaN,Is reserved.,4.0,1.058
4,05-25-batch-1,7,1,False,NaN,Tends to find fault with others.,5.0,1.214


In [39]:
df_counts = df['participant_id'].value_counts().reset_index()
failed_at_personality = df_counts[df_counts['count'] != 10]['participant_id'].values
failed_at_personality

array([24, 12])

In [40]:
df['worker_id'] = df['participant_id'].map(dict_worker_id['05-25-batch-1'])

In [42]:
df_timeouts = df[df['participant_id'].isin(failed_at_personality)][['worker_id', 'failed_reason', 'time_taken']]
df_timeouts['bonus'] = df_timeouts['time_taken'] * SECONDS_PAYMENT
df_timeouts['time_taken'] = df_timeouts['time_taken'].apply(lambda x: Utils.format_time(x))
df_timeouts

,worker_id,failed_reason,time_taken,bonus
82,67adcef788620db2ba190704,response_timeout,0:17,0.084041
161,68238de3a3ba8b99fef9b7ca,response_timeout,0:19,0.093972


In [43]:
df_timeouts['worker_id'].values

array(['67adcef788620db2ba190704', '68238de3a3ba8b99fef9b7ca'],
      dtype=object)